In [ ]:
%pip install -r BrainBERT/requirements.txt

In [ ]:
import os
import sys
module_path = os.path.abspath(os.path.join('BrainBERT'))
if module_path not in sys.path:
    sys.path.append(module_path)
import BrainBERT.models as models
from omegaconf import OmegaConf

import numpy as np
import matplotlib.pyplot as plt
from scipy import signal, stats
import torch

import warnings
os.chdir('../..')
warnings.filterwarnings('ignore')
# Firstly import the class of dataset
from Scripts.Data_Loader import EIRDataset

In [ ]:
EIR_Dataset = EIRDataset('./Generated/Data_Train/', task_type='geometric', n_jobs=72) # task type can be `geometric` or `random` or `all`

In [ ]:
def get_stft(x, fs, clip_fs=-1, normalizing=None, **kwargs):
    f, t, Zxx = signal.stft(x, fs, **kwargs)
   
    Zxx = Zxx[:clip_fs]
    f = f[:clip_fs]

    Zxx = np.abs(Zxx)
    clip = 5 #To handle boundary effects
    if normalizing=="zscore":
        Zxx = Zxx[:,clip:-clip]
        Zxx = stats.zscore(Zxx, axis=-1)
        t = t[clip:-clip]
    elif normalizing=="baselined":
        Zxx = baseline(Zxx[:,clip:-clip])
        t = t[clip:-clip]
    elif normalizing=="db":
        Zxx = np.log2(Zxx[:,clip:-clip])
        t = t[clip:-clip]

    if np.isnan(Zxx).any():
        import pdb; pdb.set_trace()

    return f, t, Zxx

In [ ]:
def build_model(cfg):
    ckpt_path = cfg.upstream_ckpt
    init_state = torch.load(ckpt_path, weights_only=False)
    upstream_cfg = init_state["model_cfg"]
    upstream = models.build_model(upstream_cfg)
    return upstream

def load_model_weights(model, states, multi_gpu):
    if multi_gpu:
        model.module.load_weights(states)
    else:
        model.load_weights(states)

In [ ]:
ckpt_path = "pretrained_weights/stft_large_pretrained.pth"
cfg = OmegaConf.create({"upstream_ckpt": ckpt_path})
model = build_model(cfg)
model.to('cuda')
init_state = torch.load(ckpt_path, weights_only=True)
load_model_weights(model, init_state['model'], False)